In [1]:
from dateutil import parser
import os
import glob
from matplotlib.dates import relativedelta
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio
from urllib.parse import urlencode
from datetime import datetime
from rasterstats import zonal_stats
import pytz
import sys

In [ ]:
import rasterio
import numpy as np
import os

year = 2025
month = 3
local_dep_dir = "/Users/cherryleheu/Documents/dews-hawaii/dews-hawaii-app/scripts/data/dependencies"
input_dir = os.path.join(local_dep_dir, "rainfall")

first_file = os.path.join(input_dir, f"rainfall_{year}_01.tif")
with rasterio.open(first_file) as src:
    meta = src.meta.copy()
    first_data = src.read(1, masked=True)
    land_mask = first_data.mask 
    ytd_sum = np.zeros(first_data.shape, dtype='float32')

for m in range(1, month + 1):
    file_path = os.path.join(input_dir, f"rainfall_{year}_{m:02d}.tif")
    with rasterio.open(file_path) as src:
        data = src.read(1).astype('float32')
        data[data == src.nodata] = 0 
        ytd_sum += data

ytd_sum[land_mask] = -9999

meta.update(dtype='float32', nodata=-9999)
output_path = os.path.join(input_dir, f'YTD_{year}_{month:02d}.tif')

with rasterio.open(output_path, 'w', **meta) as dst:
    dst.write(ytd_sum, 1)

In [22]:
#Compare to YTD climo

with rasterio.open(os.path.join(local_dep_dir, "climo", "rainfall_ytd", f"YTD_rain_month_{month:02d}.tif")) as src:
  climo_array = src.read(1).astype('float32')

with rasterio.open(os.path.join(local_dep_dir, "rainfall", output_path)) as src:
  current_array = src.read(1).astype('float32')

climo_ytd = climo_array.mean()
current_ytd = current_array.mean()
pnormal = (current_ytd/climo_ytd)*100
pnormal 

np.float32(100.128845)